# Multi-label classification

![multilabel_intro.png](https://live.staticflickr.com/65535/54927204610_28ff033051.jpg)

*Image generated by the ChatGPT image generation tool.*

## Introduction

Modern images often contain many objects and complex scenes. An example is the photograph above: we can see a man wearing a jacket, shirt, and tie, a woman in a dress with a veil, a group of people in summer clothes, and the whole scene takes place on a beach, with water and a sunset in the background.

If we were to stick to the classic approach of “one image — one label”, we would have to choose only one category: does this image depict a wedding, a man, a woman, a jacket, a dress, a beach…? In practice, this means an artificial limitation of information.

However, we do not have to limit ourselves to a single label. This is precisely why multi-label classification is used, which allows assigning multiple categories describing different objects to a single image.
Instead of choosing one label, we can say:
“This image contains instances of the classes: human, jacket, shirt, tie, dress, beach, etc.”

## Task

Your task is to define and train a neural network to perform multi-label classification. In this task, the composition of the images will be simplified compared to the example above. Only clothing items will be visible in the images, and your task is to create a model that can determine whether a given type of clothing appears in the image.

## Data

In the task you are provided with:
* a training set ($6318$ samples),
* a validation set ($702$ samples).

The test set, on which your solution will ultimately be evaluated, contains 780 samples and is not public. It was created in the same way as the validation set, so it has analogous characteristics.

Each sample is an image with dimensions $168 \times 168$ pixels. Each image is associated with a 10-element vector with values $0$ and $1$, which indicates the presence of a given class in the image. Information about which item of clothing corresponds to each index in the label vector is provided in the `LABEL_NAMES` dictionary defined in one of the code cells.

## Evaluation Criterion

The final evaluation of the task will be based on the average value of the $F1$ measure computed under the *macro* scheme.

For this task you can score between 0 and 100 points. Your final point score for the solution will be calculated according to the function below (the higher the value, the better), with additional rounding to integer values:
$$
\mathrm{score} =
\begin{cases}
    0 & \text{if } {F1}\leq 0.57 \\
    100 \times \frac{{F1}- 0.57}{0.87 - 0.57} & \text{if } 0.57 < {F1} < 0.87 \\
    100 & \text{if } {F1} \geq 0.87
\end{cases}
$$

## Constraints

- Your solution will be tested on the Competition Platform in an environment with a GPU. There is no Internet access on the Platform, however it is possible to use pretrained ResNet models (*ResNet18*, *ResNet34*, *ResNet50*) from the torchvision package, which are stored in memory as files. To use them, you must apply the same command in the code as in cases when Internet access is available.
- Evaluation of your final solution on the test data on the Competition Platform must not take longer than 2.5 minutes with a GPU.

## Submission Files

This notebook completed with your solution (model definition, model training function, and a function returning model predictions).

## Evaluation

Remember that during checking the `FINAL_EVALUATION_MODE` flag will be set to `True`.

For this task you can score between 0 and 100 points. The number of points you obtain will be calculated on the (secret) test set on the Competition Platform based on the formula mentioned above, rounded to an integer. If your solution does not meet the above criteria or does not execute correctly, you will receive 0 points for the task.

## Starter Code
In this section we initialize the environment by importing the required libraries and functions. The prepared code will help you efficiently work with the data and build an appropriate solution.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

FINAL_EVALUATION_MODE: bool = False  # Podczas sprawdzania ustawimy tę flagę na True.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

import os

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as fun
import torchvision

from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

from tqdm import tqdm

from sklearn.metrics import f1_score

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################


def seed_everything(seed: int):
    """Ustawia ziarno (seed) dla reprodukowalności wyników w Pythonie, NumPy oraz PyTorch."""

    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

# liczba klas do klasyfikacji
N_CLASSES: int = 10

# mapowanie indeksu klasy na jej nazwę
LABEL_NAMES: dict[int, str] = {
    0: "T-shirt/top",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot",
}

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

if not FINAL_EVALUATION_MODE:
    FILES: list[str] = [
        "runway-mnist/train-x.npz",
        "runway-mnist/train-y.npz",
        "runway-mnist/val-x.npz",
        "runway-mnist/val-y.npz",
    ]

    # pobierz dane na nowo, jeśli czegoś brakuje
    if not all(os.path.exists(file) for file in FILES):
        import gzip
        import tarfile
        import shutil

        if not os.path.exists("runway-mnist"):
            os.mkdir("runway-mnist")

        COMPRESSED_ARCHIVE = "runway-mnist.tar.gz"
        TAR_ARCHIVE = COMPRESSED_ARCHIVE.rstrip(".gz")
        DOWNLOAD_URL = (
            "https://drive.google.com/uc?id=1oNAFYdJyCVe3Po90KLUPxAG9HGuL7NSw"
        )

        try:
            import gdown
        except ImportError as err:
            raise RuntimeError(
                "Do pobrania zbioru danych potrzebujesz lokalnej instalacji pakietu gdown: `pip install gdown`"
            ) from err

        gdown.download(DOWNLOAD_URL, str(COMPRESSED_ARCHIVE), quiet=False)

        with gzip.open(COMPRESSED_ARCHIVE, "rb") as compressed:
            with open(TAR_ARCHIVE, "wb") as archive:
                shutil.copyfileobj(compressed, archive)

        os.remove(COMPRESSED_ARCHIVE)
        print(f"Zdekompresowano: {TAR_ARCHIVE}")

        with tarfile.open(TAR_ARCHIVE, "r") as tar:
            tar.extractall("runway-mnist")

        os.remove(TAR_ARCHIVE)
        print(f"Rozpakowano: {TAR_ARCHIVE}")

### Data loading

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

SEED: int = 42
seed_everything(SEED)


def load_x(usage) -> torch.Tensor:
    path = f"runway-mnist/{usage}-x.npz"
    return torch.tensor(np.load(path)["images"], dtype=torch.float32).unsqueeze(1)


def load_y(usage) -> torch.Tensor:
    path = f"runway-mnist/{usage}-y.npz"
    return torch.tensor(np.load(path)["labels"], dtype=torch.long)


train_dataset = TensorDataset(load_x("train"), load_y("train"))
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_dataset = TensorDataset(load_x("val"), load_y("val"))
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Evaluation function

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################


def compute_score(f1: float) -> int:
    """Oblicza wynik punktowy na podstawie wartości metryki F1."""
    lower_bound = 0.57
    upper_bound = 0.87

    if f1 <= lower_bound:
        return 0
    elif lower_bound < f1 < upper_bound:
        return int(round(100 * (f1 - lower_bound) / (upper_bound - lower_bound)))
    else:
        return 100


def evaluate_algorithm(model, predict, loader) -> float:
    """Oblicza metryki oraz ocenia Twoje rozwiązanie na ich podstawie. Zwraca obliczoną wartość metryki F1."""
    preds = []
    labels = []
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            prediction = predict(model, x.to(device)).cpu()
            preds.append(prediction)
            labels.append(y)

    predictions = torch.cat(preds)
    labels = torch.cat(labels)

    f1 = f1_score(labels.numpy(), predictions.numpy(), average="macro")
    points = compute_score(f1)
    print(f"Twój wynik F1: {f1:.3f}, co daje {points} punktów.")
    return f1

## Example solution

Below we present a simplified solution that demonstrates the basic functionality
of the notebook. It can serve as a starting point for developing your solution.

In [ ]:
######################### DO NOT CHANGE THIS CELL WHEN SUBMITTING ##########################


class NaiveSolution(nn.Module):
    """Naive solution."""

    def __init__(self):
        super().__init__()

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """This naive model predicts that all classes are present in the image."""
        BATCH_SIZE = input.size(0)
        return torch.Tensor([1] * BATCH_SIZE * N_CLASSES).reshape(BATCH_SIZE, -1)


def train_naive(_: NaiveSolution):
    """The model does not require training."""
    pass


def predict_naive(model: NaiveSolution, input: torch.Tensor) -> torch.Tensor:
    """Our model directly returns predictions, we do not process them further."""
    return model(input).to(torch.long)

In [ ]:
######################### DO NOT CHANGE THIS CELL WHEN SUBMITTING ##########################
if not FINAL_EVALUATION_MODE:
    naive_solution = NaiveSolution().to(device)
    naive_solution.train()
    train_naive(naive_solution)
    evaluate_algorithm(naive_solution, predict_naive, val_loader)

## Your solution
In the cell below you should place your solution. Introduce changes only here!

In [ ]:
class Solution(nn.Module):
    """Your solution."""

    def __init__(self):
        super().__init__()

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """Model inference"""
        BATCH_SIZE = input.size(0)
        return torch.rand(BATCH_SIZE * N_CLASSES).reshape(BATCH_SIZE, -1)


def train_solution(_: Solution):
    """Training loop for your model."""
    pass


def predict_solution(model: Solution, input: torch.Tensor) -> torch.Tensor:
    """Classification using the model.
    This function is separated to easily enable post-processing of model outputs."""
    predictions = model(input).round().to(torch.long)
    return predictions

In [ ]:
######################### DO NOT CHANGE THIS CELL WHEN SUBMITTING #########################

solution = Solution().to(device)
solution.train()

train_solution(solution)

## Evaluation

The code below will be used to evaluate the solution. After submitting the solution,
the function `evaluate_algorithm(solution, predict_solution)` will be executed, i.e.
almost identical code to the one below will be run on the test set available only to
the task evaluators.

Before submitting, make sure that the entire notebook runs from start to finish
without errors and without user intervention after executing the `Run All` command.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA #########################

if not FINAL_EVALUATION_MODE:
    evaluate_algorithm(solution, predict_solution, val_loader)